# Shot profiles: zone efficiency against the league

Where do players shoot from, and how well, compared with the league in the same zone? The league
stats API tags every shot with a court zone; this notebook takes the current WNBA season,
measures each zone's league-wide efficiency, then compares the highest-volume shooters with it.

Every number comes from `fct_shot`, the gold shot fact the site's player pages also read.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
import matplotlib.pyplot as plt
import pandas as pd
from fs import connect, describe

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
con = connect()
print(describe(con))

In [ ]:
league = 'wnba'
season = con.sql(f"select max(season) from fct_shot where league = '{league}'").fetchall()[0][0]
print(f"{league.upper()} season {season}")

league_zones = con.sql(f'''
    select shot_zone_basic as zone,
           count(*) as attempts,
           sum(case when shot_made then 1 else 0 end) as made,
           round(100.0 * sum(case when shot_made then 1 else 0 end) / count(*), 1) as fg_pct,
           round(100.0 * sum(case when shot_made then (case when is_three then 3 else 2 end) else 0 end) / count(*), 1) as points_per_100
    from fct_shot
    where league = '{league}' and season = '{season}' and shot_zone_basic <> 'Backcourt'
    group by 1 order by attempts desc
''').df()
league_zones

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(league_zones['zone'], league_zones['points_per_100'], color='#b23a0a')
ax.set_xlabel('Points per 100 attempts')
ax.set_title(f'{league.upper()} {season}: value of a shot by zone')
ax.invert_yaxis()
for i, (v, n) in enumerate(zip(league_zones['points_per_100'], league_zones['attempts'])):
    ax.text(v + 1, i, f"{v:.0f}  ({n:,} att.)", va='center', fontsize=9)
ax.set_xlim(0, league_zones['points_per_100'].max() * 1.3)
plt.tight_layout()
plt.show()

## The highest-volume shooters, zone by zone

For the ten players with the most attempts, each zone's efficiency against the league's in that
zone (zones with fewer than 15 attempts are left blank). A positive difference is a zone the player
beats the league in; the weighted sum over all zones is the player's overall edge in points per 100
attempts.

In [ ]:
players = con.sql(f'''
    with per_zone as (
        select s.player_id, s.shot_zone_basic as zone, count(*) as attempts,
               sum(case when shot_made then (case when is_three then 3 else 2 end) else 0 end) as points
        from fct_shot s
        where s.league = '{league}' and s.season = '{season}' and s.shot_zone_basic <> 'Backcourt'
        group by 1, 2
        having count(*) >= 15
    ),
    league as (
        select shot_zone_basic as zone,
               100.0 * sum(case when shot_made then (case when is_three then 3 else 2 end) else 0 end) / count(*) as league_pp100
        from fct_shot where league = '{league}' and season = '{season}' and shot_zone_basic <> 'Backcourt'
        group by 1
    ),
    top as (
        select player_id, sum(attempts) as attempts from per_zone group by 1 order by 2 desc limit 10
    )
    select coalesce(p.full_name, z.player_id) as player, z.zone, z.attempts,
           round(100.0 * z.points / z.attempts, 1) as pp100,
           round(l.league_pp100, 1) as league_pp100,
           round(100.0 * z.points / z.attempts - l.league_pp100, 1) as edge
    from per_zone z
    join top t using (player_id)
    join league l using (zone)
    left join (select league, stats_player_id, full_name from dim_player where is_current) p
        on p.league = '{league}' and p.stats_player_id = z.player_id
    order by player, z.attempts desc
''').df()
pivot = players.pivot_table(index='player', columns='zone', values='edge', aggfunc='first')
overall = (players.assign(w=players['attempts'] * players['edge'])
                  .groupby('player').agg(attempts=('attempts', 'sum'), weighted=('w', 'sum')))
overall['edge_pp100'] = (overall['weighted'] / overall['attempts']).round(1)
pivot.join(overall[['attempts', 'edge_pp100']]).sort_values('edge_pp100', ascending=False).round(1)

## Reading it

The zone table above says what a shot is worth where it is taken; the player table says who
beats that baseline and where. The same zone efficiencies, with each player's own, appear under
the shot chart on the site's player pages, so a reader can go from a name here to the chart there.